# Блок Прогноз — GAM / Generalized Additive Model

Итоговая карта строится ML-моделью GAM: расстояния до геологических факторов переводятся в proximity-признаки, затем модель учит гладкую нелинейную зависимость от каждого признака. `geo_score` как финальная ручная сумма здесь не используется.

In [ ]:
import json
import os
import re
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings("ignore")

In [ ]:
# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42
USE_SUPERVISED = True

# GAM — основная ML-модель прогноза.
# Реализация: аддитивная модель через SplineTransformer + Ridge.
# Это не ручная сумма факторов: итоговая карта строится model.predict(X).
GAM_N_KNOTS = 7          # число узлов сплайна; меньше = карта плавнее
GAM_DEGREE = 3           # кубические сплайны
GAM_ALPHA = 8.0          # L2-регуляризация; больше = меньше переобучение

# Радиус влияния известных проявлений/геохимических точек для обучающей цели.
# Это НЕ итоговый фактор, а способ построить обучающую непрерывную цель.
EVIDENCE_RADIUS_CELLS = 6.0

# Сглаживание именно ML-результата, чтобы убрать пиксельный шум.
ML_SMOOTH_PASSES = 3

# Параметры преобразования расстояний в геологические proximity-признаки.
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False

In [ ]:
# =========================
# PATHS
# =========================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "gam_main_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_gam_main.gpkg"
OUT_PNG = OUT_DIR / "forecast_gam_main.png"
OUT_COMPARE = OUT_DIR / "compare_gam_main.png"
OUT_PROX = OUT_DIR / "prox_magm_gam_main.png"
OUT_CSV = OUT_DIR / "grid_gam_main.csv"
OUT_JSON = OUT_DIR / "metrics_gam_main.json"

In [ ]:
# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    """
    Выделение наиболее перспективных зон для отображения.
    Важно: это НЕ отдельный фактор итогового прогноза.
    Итоговый прогноз уже рассчитан GAM-моделью в поле prospectivity.
    Здесь только выбираются верхние локальные области для жёлтой подсветки.
    """
    q_core = float(grid["prospectivity"].quantile(0.955))
    q_support = float(grid["prospectivity"].quantile(0.90))

    support = grid["prospectivity"] >= q_support
    local_peak = local_max_mask(grid, "prospectivity", shape)

    # Лёгкая геологическая проверка для отображения жёлтых зон:
    # пик должен иметь поддержку хотя бы одного сильного сочетания факторов.
    support_geo = (
        (grid["tect_magm_intersection"] >= grid["tect_magm_intersection"].quantile(0.70)) |
        (grid["tect_struct_intersection"] >= grid["tect_struct_intersection"].quantile(0.70)) |
        (grid["paleo_struct_intersection"] >= grid["paleo_struct_intersection"].quantile(0.70))
    )

    grid["gold_zone_raw"] = (
        (grid["prospectivity"] >= q_core) |
        (support & local_peak & support_geo)
    ).astype(int)

    grid["gold_zone"] = keep_large_components(grid, "gold_zone_raw", shape, min_cells=4).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(8, 8))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("Магматогенный фактор: proximity")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

In [ ]:
# =========================
# GRID + GEOLOGICAL ML FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

# Расстояния переводим в proximity-признаки 0..1.
# 1 = ячейка находится внутри/очень близко к фактору, 0 = далеко.
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

# Минимальный набор дополнительных геологических сочетаний как ВХОДЫ модели.
# Они не складываются вручную в финальный результат; GAM сам учит их плавный вклад.
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

FEATURE_COLS = [
    "prox_facies",
    "prox_paleo",
    "prox_struct",
    "prox_magm",
    "prox_tect1",
    "prox_tect2",
    "tect_magm_intersection",
    "tect_struct_intersection",
    "paleo_struct_intersection",
]

print("Features:", FEATURE_COLS)
print("Grid cells:", len(grid))

In [ ]:
# =========================
# GAM MODEL — ML ОСНОВА ПРОГНОЗА
# =========================

# 1) Целевая переменная для обучения.
# При наличии точек прямых поисковых признаков строим непрерывную цель:
# чем ближе ячейка к известной точке/аномалии, тем выше target.
# Это корректнее, чем грубые 0/1, потому что отсутствие точки не значит отсутствие руды.
grid["target"] = 0.0
grid["evidence_score"] = 0.0
positive_cells = []
use_supervised = False
gam_test_r2 = None
gam_test_mae = None

if USE_SUPERVISED and points is not None and len(points) > 0:
    point_union = unary_union(points.geometry)
    dist_to_evidence = np.array([geom.centroid.distance(point_union) for geom in grid.geometry], dtype=float)
    radius = CELL_SIZE * EVIDENCE_RADIUS_CELLS

    evidence_score = np.exp(-dist_to_evidence / radius)
    evidence_score = robust_normalize_01(evidence_score, 0.02, 0.995)
    grid["evidence_score"] = evidence_score

    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    if positive_cells:
        grid.loc[grid["cell_id"].isin(positive_cells), "evidence_score"] = 1.0

    grid["target"] = grid["evidence_score"]
    use_supervised = len(positive_cells) >= 3
else:
    # Запасной режим: если контрольных точек нет, GAM калибруется по экспертной близости к факторам.
    # В нормальном запуске с данными проекта этот блок обычно не нужен.
    grid["target"] = robust_normalize_01(
        0.20 * grid["prox_facies"] +
        0.20 * grid["prox_paleo"] +
        0.18 * grid["prox_struct"] +
        0.12 * grid["prox_magm"] +
        0.15 * grid["prox_tect1"] +
        0.15 * grid["prox_tect2"],
        0.02, 0.98
    )

X_all = grid[FEATURE_COLS].to_numpy(dtype=float)
y_all = grid["target"].to_numpy(dtype=float)

# Вес обучения: известные точки и ближайшая область важнее, но остальная территория тоже участвует.
sample_weight = 0.35 + 2.0 * grid["target"].to_numpy(dtype=float)
if positive_cells:
    sample_weight[grid["cell_id"].isin(positive_cells).to_numpy()] = 5.0

# 2) GAM через spline basis + Ridge.
# SplineTransformer создаёт гладкие нелинейные функции по каждому признаку отдельно.
# Ridge ограничивает амплитуду и не даёт модели переобучиться.
gam_model = make_pipeline(
    SplineTransformer(n_knots=GAM_N_KNOTS, degree=GAM_DEGREE, include_bias=False),
    StandardScaler(with_mean=False),
    Ridge(alpha=GAM_ALPHA, random_state=RANDOM_STATE),
)

if use_supervised:
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X_all, y_all, sample_weight,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=(y_all > np.quantile(y_all, 0.80)).astype(int),
    )
    gam_model.fit(X_train, y_train, ridge__sample_weight=w_train)
    pred_test = np.clip(gam_model.predict(X_test), 0, 1)
    gam_test_r2 = r2_score(y_test, pred_test)
    gam_test_mae = mean_absolute_error(y_test, pred_test)
else:
    gam_model.fit(X_all, y_all, ridge__sample_weight=sample_weight)

# Дообучаем финальную модель на всей сетке после проверки.
gam_model.fit(X_all, y_all, ridge__sample_weight=sample_weight)

grid["gam_score_raw"] = gam_model.predict(X_all)
grid["gam_score_raw"] = np.clip(grid["gam_score_raw"], 0, 1)

# Слегка усиливаем локальные максимумы и сглаживаем поверхность.
# Это постобработка ML-поверхности, а не ручная сумма факторов.
grid["gam_score_contrast"] = np.power(grid["gam_score_raw"], 1.25)
grid["gam_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "gam_score_contrast", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02, 0.98
)

# Простая интерпретация: корреляция каждого входного признака с GAM-прогнозом.
feature_importance = {}
for col in FEATURE_COLS:
    try:
        feature_importance[col] = float(np.corrcoef(grid[col].to_numpy(), grid["gam_score_sm"].to_numpy())[0, 1])
    except Exception:
        feature_importance[col] = None

print("Supervised:", use_supervised)
print("Positive cells:", len(positive_cells))
print("GAM test R2:", gam_test_r2)
print("GAM test MAE:", gam_test_mae)
print("Feature correlations with GAM score:")
for k, v in sorted(feature_importance.items(), key=lambda kv: 0 if kv[1] is None else abs(kv[1]), reverse=True):
    print(f"  {k}: {v}")

In [ ]:
# =========================
# FINAL SURFACE
# =========================

# Итоговая поверхность строится GAM-моделью.
# Здесь нет финальной формулы вида geo_score + ml_score: прогноз = GAM-предсказание.
grid["prospectivity_raw"] = grid["gam_score_sm"]

# Небольшой штраф у границ нужен только против краевых артефактов визуализации.
# Он не является отдельным прогнозным фактором.
mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(edge_near, 0.02, 0.98)

grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.015 * grid["edge_false_penalty"]
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=1)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# Для совместимости с методичкой:
# в поле prognoz меньшее значение означает более перспективную ячейку.
grid["prognoz"] = 1.0 - grid["prospectivity"]

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

In [ ]:
# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "method": "GAM via SplineTransformer + Ridge",
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "feature_cols": FEATURE_COLS,
    "gam_n_knots": GAM_N_KNOTS,
    "gam_degree": GAM_DEGREE,
    "gam_alpha": GAM_ALPHA,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(len(positive_cells)),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "point_count": int(len(points)) if points is not None else 0,
    "gam_test_r2": None if gam_test_r2 is None else float(gam_test_r2),
    "gam_test_mae": None if gam_test_mae is None else float(gam_test_mae),
    "feature_correlation_with_gam_score": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print(grid[["prospectivity", "prognoz", "gam_score_sm", "gold_zone"]].describe())